# Fase 2 · Transformación de datos en los datasets

## Objetivo

El objetivo de esta fase consiste en ejecutar los cambios y ajustes detectados durante el Análisis Exploratorio de Datos (EDA), para unificar, limpiar y transformar los datasets seleccionados en este proyecto.

En esta etapa se trabaja principalmente en:

- abordar los duplicados,
- combinar varios datasets,
- gestionar los valores nulos,
- homogenizar categorías para asegurar la integridad semántica,
- y exportar los archivos finales ya depurados.

Este proceso permitirá tener un conjunto de datos más consistentes, comparables y listos para la siguiente fase de análisis avanzado y visualización.

In [2]:
# Importación de librerías
import pandas as pd
import numpy as np
import re

# Configuración del sistema para encontrar la carpeta raíz
import sys
import os

# Esto obliga a Python a mirar una carpeta hacia atrás (donde está 'src')
sys.path.append(os.path.abspath(os.path.join('..')))
sys.path.append(os.path.abspath(os.path.join('.')))

# Importación de módulos de transformación 
from src.etl.load_data import load_friends_data_raw, load_friends_data_translated
from src.etl import transform as trans
from transformers import pipeline
from src.etl import column_standardizer as stan
from src.etl import dataset_sanitizer as pr



 
# Configuración para visualizar todas las columnas del DataFrame
pd.set_option('display.max_columns', None) 

In [3]:
dfs = load_friends_data_raw()

[2026-06-02 11:49:06] INFO - Cargando datasets desde: H:\Cursos\Adalab_Analista & IA\taller_git\friends-analytics-workflow\data_raw
[2026-06-02 11:49:06] INFO - → Cargando weddings_divorces_ross.csv...
[2026-06-02 11:49:06] INFO - → Cargando friends_cameos.csv...
[2026-06-02 11:49:06] INFO - → Cargando friends_emotions.csv...
[2026-06-02 11:49:06] INFO - → Cargando friends_episodes.csv...
[2026-06-02 11:49:06] INFO - → Cargando friends_sets.csv...
[2026-06-02 11:49:06] INFO - → Cargando friends_info.csv...
[2026-06-02 11:49:06] INFO - → Cargando friends_quotes.csv...
[2026-06-02 11:49:06] INFO - → Cargando friends.csv...
[2026-06-02 11:49:06] INFO - → Cargando phoebe_buffay_songs.csv...
[2026-06-02 11:49:06] INFO - → Cargando duck_and_chicken.csv...
[2026-06-02 11:49:06] INFO - Todos los datasets fueron cargados correctamente.


## 1. Transformación de  las variables numéricas (friends_quotes) de orden de float a entero (int) para mejorar la estructura. 

In [4]:
df_quotes = dfs["quotes"]

df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1.0,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0.0,1.0
1,Joey,1.0,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1.0,1.0
2,Chandler,1.0,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2.0,1.0
3,Phoebe,1.0,Monica Gets A Roommate,"Wait, does he eat chalk?",3.0,1.0
4,Phoebe,1.0,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4.0,1.0


In [5]:
df_quotes["quote_order"] = df_quotes["quote_order"].astype(int)
df_quotes["season"] = df_quotes["season"].astype(int)
df_quotes["episode_number"] = df_quotes["episode_number"].astype(int)


In [6]:
df_quotes.head()

,author,episode_number,episode_title,quote,quote_order,season
0,Monica,1,Monica Gets A Roommate,There's nothing to tell! He's just some guy I ...,0,1
1,Joey,1,Monica Gets A Roommate,"C'mon, you're going out with the guy! There's ...",1,1
2,Chandler,1,Monica Gets A Roommate,"All right Joey, be nice. So does he have a hum...",2,1
3,Phoebe,1,Monica Gets A Roommate,"Wait, does he eat chalk?",3,1
4,Phoebe,1,Monica Gets A Roommate,"Just, 'cause, I don't want her to go through w...",4,1


In [7]:
df_quotes.to_csv("../data_processed/friends_quotes.csv", index=False, encoding="utf-8")

## 2. Limpiar y estandarizar la columna written_by (friends_info)

In [8]:
df_info= dfs["info"]

df_info.head()

,season,episode,title,directed_by,written_by,air_date,us_views_millions,imdb_rating
0,1,1,The Pilot,James Burrows,David Crane & Marta Kauffman,1994-09-22,21.5,8.3
1,1,2,The One with the Sonogram at the End,James Burrows,David Crane & Marta Kauffman,1994-09-29,20.2,8.1
2,1,3,The One with the Thumb,James Burrows,Jeffrey Astrof & Mike Sikowitz,1994-10-06,19.5,8.2
3,1,4,The One with George Stephanopoulos,James Burrows,Alexa Junge,1994-10-13,19.7,8.1
4,1,5,The One with the East German Laundry Detergent,Pamela Fryman,Jeff Greenstein & Jeff Strauss,1994-10-20,18.6,8.5


In [9]:
trans.process_friends_writers(df_info)

¡Fichero corregido con éxito! Guardado en: H:\Cursos\Adalab_Analista & IA\taller_git\friends-analytics-workflow\data_processed\writers.csv (303 filas).


,season,episode,writer
0,1,1,David Crane
1,1,1,Marta Kauffman
2,1,2,David Crane
3,1,2,Marta Kauffman
4,1,3,Jeffrey Astrof
...,...,...,...
298,10,16,Ted Cohen
299,10,17,Marta Kauffman
300,10,17,David Crane
301,10,18,Marta Kauffman


In [10]:
df_info.drop("written_by", axis=1, inplace=True)


df_info.head(2)

,season,episode,title,directed_by,air_date,us_views_millions,imdb_rating
0,1,1,The Pilot,James Burrows,1994-09-22,21.5,8.3
1,1,2,The One with the Sonogram at the End,James Burrows,1994-09-29,20.2,8.1


In [11]:
df_info.to_csv("../data_processed/friends_info.csv", index=False, encoding="utf-8") 

#### Traducir las columnas

In [12]:
df_dac = dfs["dac"]

In [13]:
df_dac = stan.standardize_columns(df_dac)

In [14]:
df_dac.head()

,season,episode_number,animal,accion
0,3,3x21,Pollito,Joey lo compra
1,3,3x22,Pollito,Joey y Chandler cuidan de él.
2,3,3x22,Pato,Chandler lo rescata para que el pollito tenga ...
3,3,3x25,Pollito,Aparecen en el apartamento de los chicos.
4,3,3x25,Pato,Aparecen en el apartamento de los chicos.


In [15]:
df_dac.to_csv("../data_processed/duck_and_chicken.csv", index=False, encoding="utf-8")

In [16]:
df_cameos = dfs["cameos"]
df_cameos.head(1)

,Actor/Actriz,Personaje,Descripción/Temporada
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel (T8)


In [17]:
# 1. Extraemos la descripción y el número de la temporada usando Regex
# El patrón busca "T" seguido de uno o más números d+ dentro de un paréntesis
df_extracted = df_cameos["Descripción/Temporada"].str.extract(r"(?P<descripcion>.*?)\s*\(T(?P<temporada>\d+)\)")

# 2. Asignamos los resultados de vuelta a nuestro DataFrame original
df_cameos["descripcion"] = df_extracted["descripcion"]
df_cameos["temporada"] = df_extracted["temporada"]

# 3. Borramos la columna vieja que ya no necesitamos
df_cameos = df_cameos.drop(columns=["Descripción/Temporada"])

# Ver el resultado
df_cameos.head(1)

,Actor/Actriz,Personaje,descripcion,temporada
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel,8


In [18]:
df_cameos = stan.standardize_columns(df_cameos)

In [19]:
df_cameos.head()

,actor,author,description,season
0,Brad Pitt,Will Colbert,Antiguo compañero que odiaba a Rachel,8
1,Bruce Willis,Paul Stevens,Padre de Elizabeth y novio de Rachel,6
2,Julia Roberts,Susie Moss,Compañera de primaria de Chandler,2
3,Charlie Sheen,Ryan,Marinero novio de Phoebe que tiene varicela,2
4,Danny DeVito,Roy,El stripper sensible en la despedida de Phoebe,10


In [20]:
df_cameos.to_csv("../data_processed/friends_cameos.csv", index=False, encoding="utf-8")

In [21]:
df_sets = dfs["sets"]

In [22]:
df_sets = stan.standardize_columns(df_sets)

### Cargamos los datasets finales

In [23]:
dfs_finales = load_friends_data_translated()

[2026-06-02 11:49:07] INFO - Cargando datasets desde: H:\Cursos\Adalab_Analista & IA\taller_git\friends-analytics-workflow\data_translated
[2026-06-02 11:49:07] INFO - → Cargando friends_weddings_divorce_ross.csv...
[2026-06-02 11:49:07] INFO - → Cargando friends_cameos.csv...
[2026-06-02 11:49:07] INFO - → Cargando friends_emotions.csv...
[2026-06-02 11:49:07] INFO - → Cargando friends_episodes.csv...
[2026-06-02 11:49:07] INFO - → Cargando friends_sets.csv...
[2026-06-02 11:49:07] INFO - → Cargando friends_info.csv...
[2026-06-02 11:49:07] INFO - → Cargando friends_quotes.csv...
[2026-06-02 11:49:07] INFO - → Cargando friends.csv...
[2026-06-02 11:49:07] INFO - → Cargando friends_songs.csv...
[2026-06-02 11:49:07] INFO - → Cargando duck_and_chicken.csv...
[2026-06-02 11:49:07] INFO - → Cargando writers.csv...
[2026-06-02 11:49:07] INFO - Todos los datasets fueron cargados correctamente.


In [24]:
df_quotesf = dfs_finales["quotes"]

In [25]:
df_quotesf["personaje"] = df_quotesf["personaje"].str.title()

In [26]:
df_quotesf.sample(20)

,personaje,numero_episodio,titulo_episodio,cita,orden_cita,temporada
12682,Monica,5,Frank Jr.,¿Y quién lo laminó?,172,3.0
45852,Rachel,12,Joey vende con Rachel,"Oh, lirios. Joey, son mis favoritos. Gracias.",56,8.0
13342,Rachel,8,El dispositivo gigante para pinchar,O. Podríamos ponerle un sombrero en la cabeza.,67,3.0
56990,Rachel,8,El Día de Acción de Gracias tardío,¡Ay dios mío! ¡Eso es lo más espeluznante que ...,41,10.0
47180,Phoebe,17,Las hojas de té,"Oh, sí, bueno, yo también lo siento, pero ¿qué...",196,8.0
40234,Ross,15,El nuevo cerebro de Joey,¡Ya conoces la canción! ¡Canta!,313,7.0
7143,Boys,6,El bebé en el autobús,Hola.,62,2.0
55978,Chandler,4,El pastel,Te compré. ¿Cómo olvidé que eso es todo lo que...,203,10.0
14027,Ross,11,Chandler no recuerda qué hermana,"Ah, alguien está en la puerta del techo.",7,3.0
47366,Phoebe,18,En Massapequa,Entonces será mejor que lo hagas ahora.,159,8.0


#### Utilizar una funcion para detectar las lineas que no son diálogo y eliminarlas, creando un archivo nuevo y limpio.

In [27]:
pr.export_data_anomalies("../data_translated/friends_quotes.csv", "friends_quotes_errores.csv", ["personaje", "cita"] )

[INFO] No data anomalies were found in the specified target columns.


""


In [28]:
pr.sanitize_dataset_by_index("../data_translated/friends_quotes.csv", "friends_quotes_errores.csv", "friends_quotes_clean.csv", chunksize=50000)

FileNotFoundError: [ERROR] Anomalies tracking file not found at: friends_quotes_errores.csv

### Mapear archivo quotes con los nombres 


In [29]:
dicc_nombres = {
    "Phoe": "Phoebe",
    "Mnca": "Monica",
    "Rach": "Rachel",
    "Chan": "Chandler",
    "Estl": "Estelle",
    "Waiter" : "Camarera"
}

In [30]:
df_quotesf["personaje"] = df_quotesf["personaje"].apply(lambda x: dicc_nombres.get(x, x))

In [31]:
df_quotesf.tail()

,personaje,numero_episodio,titulo_episodio,cita,orden_cita,temporada
60192,Chandler,17,"El último, partes I y II","Oh, todo estará bien.",581,10.0
60193,Rachel,17,"El último, partes I y II",(llorando) ¿Tienen que ir a la nueva casa de i...,582,10.0
60194,Monica,17,"El último, partes I y II",Tenemos algo de tiempo.,583,10.0
60195,Rachel,17,"El último, partes I y II","Bien, ¿deberíamos tomar un poco de café?",584,10.0
60196,Chandler,17,"El último, partes I y II",Seguro. ¿Dónde?,585,10.0


In [32]:
df_quotesf.to_csv("../data_translated/friends_quotes.csv", index=False, encoding="utf-8")

### Normalizamos las columnas del fichero friends

In [33]:
df_friendsf = dfs_finales["friends"]

In [34]:
df_friendsf = stan.standardize_columns(df_friendsf)

In [35]:
df_friendsf.head()

,texto,author,season,numero_episodio,escena,orden_intervencion
0,¡No hay nada que contar! ¡Es sólo un tipo con ...,Monica Geller,1,1,1,1
1,"¡Vamos, vas a salir con el chico! ¡Tiene que h...",Joey Tribbiani,1,1,1,2
2,"Muy bien Joey, sé amable. Entonces ¿tiene joro...",Chandler Bing,1,1,1,3
3,"Espera, ¿come tiza?",Phoebe Buffay,1,1,1,4
4,"(Todos se quedan mirando, desconcertados.)",Scene Directions,1,1,1,5


In [ ]:
df_friendsf.to_csv("../data_translated/friends.csv", index=False, encoding="utf-8")

### normalizar fecha_estreno del fichero info aaaa-mm-dd ponerlo como dd-mm-aaaa

In [37]:
df_infof = dfs_finales["info"]

In [38]:
df_infof = trans.cambiar_formato_fecha(df_infof, "fecha_estreno")

df_infof.head()

,temporada,episodio,titulo,director,fecha_estreno,audiencia_millones,nota_imdb
0,1,1,El de Monica consigue una compañera,James Burrows,22/09/1994,21.5,8.3
1,1,2,El del sonograma al final,James Burrows,29/09/1994,20.2,8.1
2,1,3,El del pulgar,James Burrows,06/10/1994,19.5,8.2
3,1,4,El de George Stephanopoulos,James Burrows,13/10/1994,19.7,8.1
4,1,5,El del detergente de Alemania Oriental,Pamela Fryman,20/10/1994,18.6,8.5


In [40]:
 
# 3. Guardar el resultado en un nuevo archivo para no machacar el original
df_infof.to_csv("../data_translated/friends.csv", index=False)
 